In [ ]:
import cv2
import numpy as np
from PIL import Image

In [ ]:
def cartoonize_image(img, k=10, it = 10, t1 = 150, t2 = 255, ks = 1 , kc=1):
    # Apply bilateral filter to smooth the image
    img_color = cv2.bilateralFilter(img, d=9, sigmaColor=75, sigmaSpace=75)
    # Convert to grayscale
    img_gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    # Apply Gaussian Blur
    img_blur = cv2.GaussianBlur(img_gray, (kc, kc), 0)
    # Detect edges using Canny edge detection
    edges = cv2.Canny(img_blur, threshold1=t1, threshold2=t2)
    # Dilate the edges to make them more prominent
    kernel = np.ones((ks, ks), np.uint8)
    edges = cv2.dilate(edges, kernel, iterations=1)
    # Invert the edges
    edges = cv2.bitwise_not(edges)
    # Convert edges back to color, so we can combine with color image
    edges_colored = cv2.cvtColor(edges, cv2.COLOR_GRAY2RGB)
    # Perform K-means clustering
    img_data = np.float32(img_color).reshape((-1, 3))
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, it, 0.2)
    _, labels, centers = cv2.kmeans(img_data, k, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS)
    centers = np.uint8(centers)
    img_clustered = centers[labels.flatten()]
    img_clustered = img_clustered.reshape(img_color.shape)
    # Combine edge and clustered image
    cartoon = cv2.bitwise_and(img_clustered, edges_colored)
    return cartoon

In [ ]:
def apply_palette(img, palette=[
    (0, 0, 0),        # Black
    (255, 255, 255),  # White
    (255, 0, 0),      # Red
    (0, 255, 0),      # Green
    (0, 0, 255),      # Blue
]):
    # Convert input to PIL if needed
    if not isinstance(img, Image.Image):
        img = Image.fromarray(img)
    img = img.convert("RGB")
    # Create palette image
    palette_img = Image.new("P", (1, 1))
    palette_img.putpalette(sum(palette, ()))
    # Quantize
    quantized = img.quantize(palette=palette_img).convert("RGB")
    # Convert back to numpy (RGB)
    return np.array(quantized)

In [ ]:
def color_limit(img,N_colors=20):
    # Convert BGR (OpenCV format) to RGB (PIL format)
    image_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    # Convert NumPy array (RGB) to PIL Image
    pil_image = Image.fromarray(image_rgb)
    # Apply the quantize method in PIL
    quantized_image = pil_image.quantize(N_colors)
    # Convert the quantized PIL image back to NumPy array (RGB format)
    image_rgb_back = np.array(quantized_image.convert('RGB'))  # Convert quantized image back to RGB
    #Convert RGB back to BGR for displaying with OpenCV
    image_bgr_back = cv2.cvtColor(image_rgb_back, cv2.COLOR_RGB2BGR)
    return image_bgr_back

In [ ]:
def cartoonize_vid(F,t1 = 150, t2 = 255, ks = 1 , kc=1):
    C = []
    centers = np.uint8(np.asarray([[r,g,b] for r in range(0,255,40) for g in range(0,255,40) for b in range(0,255,40)]))
    It = 0
    for f in F:
        It = It+1
        print(It,'/',len(F),end='\r')
        img_color = cv2.bilateralFilter(f, d=9, sigmaColor=75, sigmaSpace=75)
        # Convert to grayscale
        img_gray = cv2.cvtColor(f, cv2.COLOR_RGB2GRAY)
        # Apply Gaussian Blur
        img_blur = cv2.GaussianBlur(img_gray, (kc, kc), 0)
        # Detect edges using Canny edge detection
        edges = cv2.Canny(img_blur, threshold1=t1, threshold2=t2)
        # Dilate the edges to make them more prominent
        kernel = np.ones((ks, ks), np.uint8)
        edges = cv2.dilate(edges, kernel, iterations=1)
        # Invert the edges
        edges = cv2.bitwise_not(edges)
        # Convert edges back to color, so we can combine with color image
        edges_colored = cv2.cvtColor(edges, cv2.COLOR_GRAY2RGB)
        # Perform K-means clustering
        img_data = np.float32(img_color).reshape((-1, 3))

        distances = np.linalg.norm(img_data[:, np.newaxis] - centers, axis=2)
        closest_clusters = np.argmin(distances, axis=1)

        # Map the pixels in the second image to the closest cluster centers
        segmented_pixels2 = centers[closest_clusters]
        segmented_image2 = segmented_pixels2.reshape(f.shape)

        # Combine edge and clustered image
        cartoon = cv2.bitwise_and(segmented_image2, edges_colored)
        C.append(cartoon)
    return C